In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/insurance.csv')
df.head()

#EDA

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
numeric_col = ['age','bmi','children','charges']
for col in numeric_col:
   plt.figure(figsize = (10,6))
   sns.histplot(df[col], kde=True , bins=20)

In [ ]:
sns.countplot(x = df['children'])

In [ ]:
sns.countplot(x = df['sex'])

In [ ]:
sns.countplot(x = df['smoker'])

In [ ]:
for col in numeric_col:
  plt.figure(figsize=(10,6))
  sns.boxplot(x = df[col])

In [ ]:
plt.figure(figsize=(10,6))
sns.heatmap(df.corr(numeric_only=True), annot=True)

#Data Cleaning and preprocessing

In [ ]:
df_copy = df.copy()
df.head()

In [ ]:
df_copy.reset_index
df_copy.shape

In [ ]:
df_copy.drop_duplicates(inplace=True)
df_copy.shape

In [ ]:
df_copy.isnull().sum()

In [ ]:
df_copy.dtypes

In [ ]:
print(df_copy['sex'].value_counts())
print(df_copy['smoker'].value_counts())
print(df_copy['region'].value_counts())

In [ ]:
#label encoding
df_copy['sex'] = df_copy['sex'].map({"male" : 0 , "female":1})
df_copy['smoker'] = df_copy['smoker'].map({"yes":0 , "no":1})

#one hot encoding
df_copy = pd.get_dummies(df_copy , columns=['region'],drop_first=True,dtype =float)

In [ ]:
df_copy.head()

In [ ]:
df_copy.rename(columns={'sex' : 'Gender'},inplace=True)
df_copy.head()

In [ ]:
df_copy = df_copy.astype(int)

In [ ]:
df_copy

#feature Engineering and extracton

In [ ]:
sns.histplot(df['bmi'])

In [ ]:
ages = np.array([5, 15, 25, 35, 45, 55, 65, 75])

bins = [0,10,20,50,60]
# To match the 4 intervals created by 5 bins, we need 4 labels.
label = ['child' , 'young' , 'youth' , 'middle']

new = pd.cut(ages , bins=bins ,labels=label)
print(new)

In [ ]:
df_copy['bmi_catagorey'] = pd.cut(
    df_copy['bmi'],
    bins=[0,18.5,24.9,29.9 , float('inf')],
    labels=['under_weight','Normal','Over_weight','obeese']
)
df_copy

In [ ]:
df_copy = pd.get_dummies(df_copy , columns=['bmi_catagorey'],drop_first=True,dtype =float)
df_copy=df_copy.astype(int)
df_copy.head()

In [ ]:
df_copy.columns

In [ ]:
from sklearn.preprocessing import StandardScaler
col = ['age','bmi','children']
scale = StandardScaler()

df_copy[col] = scale.fit_transform(df_copy[col])

In [ ]:
df_copy

#extraction


In [ ]:
from scipy.stats import pearsonr
selected_column = ['age', 'Gender', 'bmi', 'children', 'smoker', 'charges','region_northwest', 'region_southeast', 'region_southwest','bmi_catagorey_Normal', 'bmi_catagorey_Over_weight','bmi_catagorey_obeese']

corelation = {
     feature:pearsonr(df_copy[feature],df_copy['charges'])[0]
     for feature in selected_column
}

coorelation_df = pd.DataFrame(list(corelation.items()),columns=['feature','pearson corealtion'])
coorelation_df.sort_values(by='pearson corealtion',ascending=False)

In [ ]:
cat_selected=['Gender','smoker','region_northwest', 'region_southeast', 'region_southwest','bmi_catagorey_Normal', 'bmi_catagorey_Over_weight','bmi_catagorey_obeese']



In [ ]:
from scipy.stats import chi2_contingency

alpha = 0.05
df_copy['charges bin']=pd.qcut(df_copy['charges'],q=4,labels=['Low', 'Medium', 'High', 'Very High'])
chi2_results={}

for col in cat_selected:
  contingency = pd.crosstab(df_copy[col],df_copy['charges bin'])

  chi2_stat , p_val ,_,_= chi2_contingency(contingency)
  decision = 'Reject Null (keep Feature)' if p_val < alpha else 'Accpet Null (drop feature)'
  chi2_results[col] = {
      'chi2_statisticts':chi2_stat,
      'p_value': p_val,
      'Decision': decision
  }

chi2_df = pd.DataFrame(chi2_results).T
chi2_df = chi2_df.sort_values(by = 'p_value')
chi2_df


In [ ]:
final_df = df_copy[['age', 'Gender', 'bmi', 'children', 'smoker', 'charges',
'region_southeast','bmi_catagorey_obeese']]
final_df

In [ ]:
final_df.to_csv('insurance valuable.csv')